# **ginger**

In [27]:
import sys
import os

if "google.colab" in sys.modules:
    import getpass

    token = getpass.getpass()

    repo_url = f"https://{token}@github.com/Mango-Cats/ginger.git"

    if os.path.exists("ginger"):
        %cd ginger
        !git pull origin main
    else:
        !git clone {repo_url}
        %cd ginger

    try:
        %load_ext autoreload
        %autoreload 2
    except Exception as e:
        print("autoreload not available:", e)

    !pip install -e . --no-deps

Cloning into 'ginger'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 78 (delta 36), reused 56 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (78/78), 254.75 KiB | 1.25 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/ginger/ginger/ginger/ginger
autoreload not available: No module named 'imp'
Obtaining file:///content/ginger/ginger/ginger/ginger
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ginger (pyproject.toml) ... done
  Created wheel for ginger: filename=ginger-0.1.0-0.editable-py3-none-any.whl size=5482 sha256=38a20b1e12630e4887ca76c421121a24388c1790420176ee34a4555389a02b5f
  Stored in directory: /tmp/pip-ephem-wheel-cache-wz482jbc/wheels/3e/43/81/3f7bee1f1e667276c29bdaa135

In [28]:
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)
print(DEVICE)

2.10.0+cu128
cuda


## **Data Loading & Preprocessing**

In [29]:
from src.dataman import create_dataloaders
from src.dataman.constants import PAD_IDX
from pathlib import Path

In [30]:
DATA_PATH = Path("data/wikipron_tagalog.tsv")
BATCH_SIZE = 64

train_loader, val_loader, test_loader, grapheme_vocab, phoneme_vocab = create_dataloaders(
    DATA_PATH,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

print(f"Train samples: {len(train_loader.dataset)}")
print(f"Val samples: {len(val_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}")
print(f"Grapheme vocab size: {len(grapheme_vocab)}")
print(f"Phoneme vocab size: {len(phoneme_vocab)}")

Train samples: 14590
Val samples: 1824
Test samples: 1824
Grapheme vocab size: 32
Phoneme vocab size: 34


## **Model Initialization**

In [31]:
from src.architecture.transformer import Transformer
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim import AdamW
import torch.nn as nn

In [32]:
EMBEDDING_DIM = 128
NUM_ATTENTION_HEADS = 4
NUM_ENCODER_LAYERS = 4
NUM_DECODER_LAYERS = 4
FEEDFORWARD_DIM = 512
DROPOUT_RATE = 0.1
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 100 

model = Transformer(
    num_graphemes=len(grapheme_vocab),
    num_phonemes=len(phoneme_vocab),
    embedding_dim=EMBEDDING_DIM,
    num_attention_heads=NUM_ATTENTION_HEADS,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    feedforward_dim=FEEDFORWARD_DIM,
    dropout_rate=DROPOUT_RATE,
).to(DEVICE)

model = torch.compile(model)

CRITERION = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
OPTIMIZER = AdamW(params=model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
SCHEDULER = OneCycleLR(
    optimizer=OPTIMIZER,
    max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.1 
)
SCALER = torch.cuda.amp.GradScaler()
PATIENCE = 7

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(
/tmp/ipykernel_16149/2010783063.py:33: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  SCALER = torch.cuda.amp.GradScaler()


## **Training**

In [ ]:
from src.trainer import train_model

In [ ]:
results = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=CRITERION,
    optimizer=OPTIMIZER,
    scheduler=SCHEDULER,
    scaler=SCALER,
    device=DEVICE,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
)


Starting training for 100 epochs...

Epoch   5/100 | Train Loss: 1.6532 | Val Loss: 1.5508 (Best)
Epoch  10/100 | Train Loss: 1.3389 | Val Loss: 1.3014 (Best)
Epoch  15/100 | Train Loss: 1.2057 | Val Loss: 1.2034 (No improvement for 1 epochs)
Epoch  20/100 | Train Loss: 1.1290 | Val Loss: 1.1602 (No improvement for 2 epochs)
Epoch  25/100 | Train Loss: 1.0847 | Val Loss: 1.1656 (No improvement for 7 epochs)

Stopping early (patience=7)
Best validation loss: 1.1227
Restoring best model...


## **Inference**

In [38]:
from src.evaluation.evaluate_model import evaluate_model

ModuleNotFoundError: No module named 'src.evaluation.evaluate_model'

In [ ]:
eval_results = evaluate_model(
    model=model,
    test_loader=test_loader,
    grapheme_vocab=grapheme_vocab,
    phoneme_vocab=phoneme_vocab,
    device=DEVICE,
)